# Two-sample and paired t-tests
Desktop against mobile users, Titanic male against female ages, and a weight-loss program, with a check of why pairing helps.
Note: website_time.csv holds simulated values built to match the summary numbers of the example.

In [1]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Desktop and mobile users: independent two-sample t-test

In [2]:
web = pd.read_csv("data/website_time.csv")
desktop = web.loc[web.device == "desktop", "minutes"]
mobile = web.loc[web.device == "mobile", "minutes"]
print(web.groupby("device").minutes.agg(["count", "mean", "std"]).round(2))
# assumptions: normality of each group, equal variances
print("Shapiro p:", round(stats.shapiro(desktop).pvalue, 2), round(stats.shapiro(mobile).pvalue, 2))
print("Levene p: ", round(stats.levene(desktop, mobile).pvalue, 2))

         count  mean  std
device                   
desktop     30  18.5  3.5
mobile      30  14.3  2.7
Shapiro p: 0.51 0.77
Levene p:  0.3


In [3]:
# the formula by hand
se = np.sqrt(3.5 ** 2 / 30 + 2.7 ** 2 / 30)
t = (18.5 - 14.3) / se
print(f"SE = {se:.3f}, t = {t:.2f}, p = {2 * stats.t.sf(t, df=58):.2g}")
# scipy: Student's test (equal variances) and Welch's test
print(stats.ttest_ind(desktop, mobile))
print(stats.ttest_ind(desktop, mobile, equal_var=False))

SE = 0.807, t = 5.20, p = 2.7e-06
TtestResult(statistic=np.float64(5.204134418413101), pvalue=np.float64(2.6702108031356428e-06), df=np.float64(58.0))
TtestResult(statistic=np.float64(5.204134418413101), pvalue=np.float64(3.039301744782554e-06), df=np.float64(54.488994496993584))


## 2. Titanic: are men older than women?

In [4]:
titanic = pd.concat([pd.read_csv("data/titanic_train.csv"), pd.read_csv("data/titanic_test.csv")])
male = titanic.loc[titanic.Sex == "male", "Age"].dropna().reset_index(drop=True)
female = titanic.loc[titanic.Sex == "female", "Age"].dropna().reset_index(drop=True)
print(len(male), "men,", len(female), "women with known age")
# one random sample of 25 from each group
s_male, s_female = male.sample(25, random_state=5), female.sample(25, random_state=5)
print(f"sample means {s_male.mean():.1f} and {s_female.mean():.1f}")
print("Shapiro p:", round(stats.shapiro(s_male).pvalue, 2), round(stats.shapiro(s_female).pvalue, 2))
print("Levene p: ", round(stats.levene(s_male, s_female).pvalue, 2))
res = stats.ttest_ind(s_male, s_female, alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.2f}")
# the truth: population means
print(f"population means: men {male.mean():.2f}, women {female.mean():.2f}")

658 men, 388 women with known age
sample means 32.9 and 31.8
Shapiro p: 0.78 0.52
Levene p:  0.89
t = 0.25, p = 0.40
population means: men 30.59, women 28.69


In [5]:
# power: how often does a pair of samples of 25 detect the real difference?
rej = [stats.ttest_ind(male.sample(25, random_state=i), female.sample(25, random_state=10_000 + i),
                       alternative="greater").pvalue <= 0.05 for i in range(4000)]
print("share of sample pairs that reject H0:", np.mean(rej))

share of sample pairs that reject H0: 0.10475


## 3. Weight-loss program: paired t-test

In [6]:
w = pd.read_csv("data/weight_loss.csv")
d = w["before"] - w["after"]
print("differences:", d.tolist())
print(f"mean {d.mean():.2f}, sd {d.std():.2f}, Shapiro p {stats.shapiro(d).pvalue:.2f}")
# H1: the program reduces weight, i.e. before - after > 0
res = stats.ttest_rel(w["before"], w["after"], alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.2f}")
# the same as a one-sample t-test of the differences against 0
print(stats.ttest_1samp(d, 0, alternative="greater"))

differences: [2, -1, -6, 1, -3, 2, -1, -1, -2, 2, -1, -2, 2, -2, 3]
mean -0.47, sd 2.45, Shapiro p 0.16
t = -0.74, p = 0.76
TtestResult(statistic=np.float64(-0.739038797323283), pvalue=np.float64(0.7639533793596126), df=np.int64(14))


## 4. Why pairing helps

In [7]:
se_paired = d.std() / np.sqrt(len(d))
se_indep = np.sqrt(w["before"].var() / 15 + w["after"].var() / 15)
print(f"paired SE {se_paired:.2f}, independent SE {se_indep:.2f}")
# suppose everyone ended 2 kg lighter than recorded
after2 = w["after"] - 2
print("paired:     ", stats.ttest_rel(w["before"], after2, alternative="greater"))
print("independent:", stats.ttest_ind(w["before"], after2, alternative="greater"))

paired SE 0.63, independent SE 2.75
paired:      TtestResult(statistic=np.float64(2.428270334062216), pvalue=np.float64(0.01461995426239161), df=np.int64(14))
independent: TtestResult(statistic=np.float64(0.557504090398277), pvalue=np.float64(0.29080656973661156), df=np.float64(28.0))


In [8]:
# not a lucky table: simulate 2000 groups of 15 people like ours
# weights spread by 8 kg, true loss 1.5 kg, each person's change varies by 2.5 kg
rng = np.random.default_rng(0)
p_pair, p_ind = [], []
for _ in range(2000):
    before = rng.normal(80, 8, 15)
    after = before - 1.5 + rng.normal(0, 2.5, 15)
    p_pair.append(stats.ttest_rel(before, after, alternative="greater").pvalue)
    p_ind.append(stats.ttest_ind(before, after, alternative="greater").pvalue)
print(f"median p: paired {np.median(p_pair):.3f}, independent {np.median(p_ind):.3f}")

median p: paired 0.017, independent 0.305
